In [ ]:
import sys, os, glob, shutil, time, json, gc
import numpy as np, pandas as pd, torch
t0 = time.perf_counter()
def log(m): print(f"[{time.perf_counter()-t0:7.0f}с] {m}", flush=True)
name = torch.cuda.get_device_name(0); log(f"GPU: {name}")
if "T4" not in name and "L4" not in name and "A100" not in name:
    raise SystemExit(f"нужна T4, выдали {name}")
code = os.path.dirname(glob.glob("/kaggle/input/**/cross_encoder.py", recursive=True)[0])
os.makedirs("/kaggle/working/src", exist_ok=True)
for p in glob.glob(code + "/*.py"): shutil.copy(p, "/kaggle/working/src/")
open("/kaggle/working/src/__init__.py", "a").close()
os.chdir("/kaggle/working"); sys.path.insert(0, "/kaggle/working")
from src.cross_encoder import build_product_texts
from transformers import AutoModelForSequenceClassification, AutoTokenizer

pack = os.path.dirname(glob.glob("/kaggle/input/**/llm_train.parquet", recursive=True)[0])
big = os.path.dirname(glob.glob("/kaggle/input/**/llm_pairs_2m.parquet", recursive=True)[0])
seen = pd.read_parquet(pack + "/llm_train.parquet", columns=["id1", "id2"])
pairs = pd.read_parquet(big + "/llm_pairs_2m.parquet")
items = pd.read_parquet(big + "/llm_items_2m.parquet")
log(f"всего пар {len(pairs):,}, энкодеры видели {len(seen):,}")

key = lambda d: pd.MultiIndex.from_arrays([d.id1.to_numpy(), d.id2.to_numpy()])
clean = pairs[~key(pairs).isin(key(seen))].reset_index(drop=True)
log(f"не виденных энкодерами: {len(clean):,}")
LIMIT = 400_000
if len(clean) > LIMIT:
    clean = clean.sample(LIMIT, random_state=2026).reset_index(drop=True)
used = pd.unique(np.concatenate([clean.id1.to_numpy(), clean.id2.to_numpy()]))
items = items[items.id.isin(used)].reset_index(drop=True)
log(f"берём {len(clean):,} пар, карточек {len(items):,}, доля+ {(clean.label>0).mean():.3f}")

texts = build_product_texts(items, "compact")
left = clean.id1.map(texts).fillna("").astype(str).to_numpy()
right = clean.id2.map(texts).fillna("").astype(str).to_numpy()
order = np.argsort(np.fromiter((len(a)+len(b) for a, b in zip(left, right)), dtype=np.int32, count=len(left)))
log("тексты собраны")

# Веса лежат в датасете плоско, с префиксом имени модели: Kaggle не хранит вложенные
# каталоги, а `from_pretrained` требует именно каталог. Раскладываем обратно.
flat = os.path.dirname(glob.glob("/kaggle/input/**/ce_relaxed__model.safetensors", recursive=True)[0])
MODELS = ("ce_relaxed", "ce_combo", "ce_spec", "ce_self")
for tag in MODELS:
    os.makedirs(f"/kaggle/working/enc/{tag}", exist_ok=True)
    for f in ("model.safetensors", "config.json", "tokenizer.json",
              "tokenizer_config.json", "inference_config.json"):
        os.symlink(f"{flat}/{tag}__{f}", f"/kaggle/working/enc/{tag}/{f}")
out = {}
for tag in MODELS:
    path = f"/kaggle/working/enc/{tag}"
    t = time.perf_counter()
    tok = AutoTokenizer.from_pretrained(path, local_files_only=True)
    model = AutoModelForSequenceClassification.from_pretrained(
        path, local_files_only=True, dtype=torch.float16).cuda().eval()
    scores = np.empty(len(clean), dtype=np.float32)
    BATCH = 256
    with torch.inference_mode():
        for start in range(0, len(order), BATCH):
            rows = order[start:start+BATCH]
            enc = tok(left[rows].tolist(), right[rows].tolist(), padding=True, truncation=True,
                      max_length=256, pad_to_multiple_of=8, return_tensors="pt").to("cuda")
            scores[rows] = model(**enc).logits.squeeze(-1).float().cpu().numpy()
    out[tag] = scores
    np.save(f"/kaggle/working/{tag}.npy", scores)
    del model; torch.cuda.empty_cache(); gc.collect()
    log(f"  {tag}: {time.perf_counter()-t:.0f}с ({len(clean)/(time.perf_counter()-t):.0f} пар/с)")

clean.to_parquet("/kaggle/working/clean_pairs.parquet", index=False)
items.to_parquet("/kaggle/working/clean_items.parquet", index=False)
log("готово: пары, карточки и четыре набора скоров сохранены")
